In [1]:
import pickle
import torch
import gradio as gr
from transformers import BertForSequenceClassification

D:\college\SixthSem\NLP\mid\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load models
with open('deploy/variables.pkl', 'rb') as f:
    naive_model, logistic_model, tfidf_vectorizer, tokenizer = pickle.load(f)

bert_model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
bert_model.load_state_dict(torch.load('deploy/bert_model.pth', map_location=torch.device('cpu')))
bert_model.eval()


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [4]:
def predict_response(text, history):  
    # Traditional models
    tfidf_input = tfidf_vectorizer.transform([text])
    naive_prediction = naive_model.predict(tfidf_input)[0]
    logistic_prediction = logistic_model.predict(tfidf_input)[0]

    # BERT model
    bert_model.eval()
    encoded_input = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        output = bert_model(**encoded_input)
        bert_prediction = torch.argmax(output.logits, axis=1).item()

    result = []
    result.append(f"Naive Bayes: {'Spam' if naive_prediction == 1 else 'Ham'}")
    result.append(f"Logistic Regression: {'Spam' if logistic_prediction == 1 else 'Ham'}")
    result.append(f"BERT: {'Spam' if bert_prediction == 1 else 'Ham'}")

    return "\n".join(result)

chat = gr.ChatInterface(
    fn=predict_response,
    title="Spam Detector Chat",
    description="Send a message and see how different models classify it.",
    textbox=gr.Textbox(placeholder="Type an email or message here...", lines=3),
)

chat.launch(share=True)


D:\college\SixthSem\NLP\mid\venv\Lib\site-packages\gradio\chat_interface.py:338: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
